In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import * 
spark=SparkSession.builder\
.appName("Questions")\
.master("spark://172.18.0.2:7077")\
.getOrCreate()

spark

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/07/08 00:07:37 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/07/08 00:07:38 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


In [11]:
emp_df = spark.read.format("csv").option("header" , True).option("inferSchema" , True).load("/opt/spark-data/employee.csv")
emp_df.show()

+------+-------+----------+------+----------+
|emp_id|   name|department|salary|experience|
+------+-------+----------+------+----------+
|   101|   John|        IT| 75000|         3|
|   102|  Alice|        HR| 50000|         5|
|   103|    Bob|   Finance| 90000|         8|
|   104|  David|        IT| 45000|         2|
|   105|   Emma| Marketing| 65000|         6|
|   106|Michael|   Finance| 30000|         1|
|   107| Sophia|        HR| 85000|         7|
|   108|  James|        IT| 55000|         4|
|   109| Olivia| Marketing| 78000|         5|
|   110|William|   Finance| 95000|        10|
|   111|   Noah|        IT| 40000|         2|
|   112|    Ava|        HR| 72000|         6|
|   113|  Ethan|   Finance| 88000|         9|
|   114|    Mia| Marketing| 52000|         3|
|   115|   Liam|        IT| 25000|         1|
+------+-------+----------+------+----------+



In [14]:
# salary >= 80000 → "High"
# salary between 50000 and 80000 → "Medium"
# salary < 50000 → "Low"

emp_new = emp_df.withColumn("salaryLevel", 
                  when(col('salary')>=80000 , "High").when(col('salary')>=50000 , "Medium").otherwise("low"))
emp_new.show()

+------+-------+----------+------+----------+-----------+
|emp_id|   name|department|salary|experience|salaryLevel|
+------+-------+----------+------+----------+-----------+
|   101|   John|        IT| 75000|         3|     Medium|
|   102|  Alice|        HR| 50000|         5|     Medium|
|   103|    Bob|   Finance| 90000|         8|       High|
|   104|  David|        IT| 45000|         2|        low|
|   105|   Emma| Marketing| 65000|         6|     Medium|
|   106|Michael|   Finance| 30000|         1|        low|
|   107| Sophia|        HR| 85000|         7|       High|
|   108|  James|        IT| 55000|         4|     Medium|
|   109| Olivia| Marketing| 78000|         5|     Medium|
|   110|William|   Finance| 95000|        10|       High|
|   111|   Noah|        IT| 40000|         2|        low|
|   112|    Ava|        HR| 72000|         6|     Medium|
|   113|  Ethan|   Finance| 88000|         9|       High|
|   114|    Mia| Marketing| 52000|         3|     Medium|
|   115|   Lia

In [18]:
emp_new = emp_new.withColumn('new_name' , upper(col('name')))
emp_new = emp_new.drop(col('name')).withColumnRenamed('new_name', 'name').withColumn('full_name' , concat(col('first_name') , lit(' ') , col('last_name')))
emp_new.show()

+------+----------+------+----------+-----------+-------+
|emp_id|department|salary|experience|salaryLevel|   name|
+------+----------+------+----------+-----------+-------+
|   101|        IT| 75000|         3|     Medium|   JOHN|
|   102|        HR| 50000|         5|     Medium|  ALICE|
|   103|   Finance| 90000|         8|       High|    BOB|
|   104|        IT| 45000|         2|        low|  DAVID|
|   105| Marketing| 65000|         6|     Medium|   EMMA|
|   106|   Finance| 30000|         1|        low|MICHAEL|
|   107|        HR| 85000|         7|       High| SOPHIA|
|   108|        IT| 55000|         4|     Medium|  JAMES|
|   109| Marketing| 78000|         5|     Medium| OLIVIA|
|   110|   Finance| 95000|        10|       High|WILLIAM|
|   111|        IT| 40000|         2|        low|   NOAH|
|   112|        HR| 72000|         6|     Medium|    AVA|
|   113|   Finance| 88000|         9|       High|  ETHAN|
|   114| Marketing| 52000|         3|     Medium|    MIA|
|   115|      

In [20]:
product_df = spark.read.format("csv").option("header", True).option("inferSchema" , True).load("/opt/spark-data/product.csv")
product_df.show()


+----------+------------+-----+
|product_id|product_name|price|
+----------+------------+-----+
|         1|      Laptop|75000|
|         2|       Mouse| 1200|
|         3|    Keyboard| 2500|
|         4|     Monitor|15000|
|         5|  Headphones| 3500|
|         6|      Mobile|45000|
|         7|      Tablet|28000|
|         8|     Printer| 9000|
|         9|      Camera|55000|
|        10|  Smartwatch| 8000|
+----------+------------+-----+



In [21]:
product_df.printSchema()

root
 |-- product_id: integer (nullable = true)
 |-- product_name: string (nullable = true)
 |-- price: integer (nullable = true)



In [23]:
product_df.withColumn('price'  , integerType(col('price')))

NameError: name 'integerType' is not defined

In [24]:
product_df.withColumn('gst' , col('price')*0.18).show()


+----------+------------+-----+-------+
|product_id|product_name|price|    gst|
+----------+------------+-----+-------+
|         1|      Laptop|75000|13500.0|
|         2|       Mouse| 1200|  216.0|
|         3|    Keyboard| 2500|  450.0|
|         4|     Monitor|15000| 2700.0|
|         5|  Headphones| 3500|  630.0|
|         6|      Mobile|45000| 8100.0|
|         7|      Tablet|28000| 5040.0|
|         8|     Printer| 9000| 1620.0|
|         9|      Camera|55000| 9900.0|
|        10|  Smartwatch| 8000| 1440.0|
+----------+------------+-----+-------+



In [25]:
transaction_df = spark.read.format("csv").option("header", True).option("inferSchema" , True).load("/opt/spark-data/transactions.csv")
transaction_df.show()

+--------------+-------+------+-------+
|transaction_id|user_id|amount| status|
+--------------+-------+------+-------+
|             1|    101|   500|SUCCESS|
|             2|    102|   200| FAILED|
|             3|    103|   800|SUCCESS|
|             4|    104|   100|PENDING|
|             5|    105|  1200|SUCCESS|
|             6|    106|   250| FAILED|
|             7|    107|   650|SUCCESS|
|             8|    108|   300|PENDING|
|             9|    109|   950|SUCCESS|
|            10|    110|   150|SUCCESS|
|            11|    111|   700|SUCCESS|
|            12|    112|   400| FAILED|
|            13|    113|  1000|SUCCESS|
|            14|    114|    50|PENDING|
|            15|    115|   850|SUCCESS|
+--------------+-------+------+-------+



In [29]:
txn_df= transaction_df.filter((col('status')=='SUCCESS') & (col('amount')>=300)).withColumn('transaction_type' , when(col('amount')>700 , 'Large').otherwise("Normal"))
txn_df.show()

+--------------+-------+------+-------+----------------+
|transaction_id|user_id|amount| status|transaction_type|
+--------------+-------+------+-------+----------------+
|             1|    101|   500|SUCCESS|          Normal|
|             3|    103|   800|SUCCESS|           Large|
|             5|    105|  1200|SUCCESS|           Large|
|             7|    107|   650|SUCCESS|          Normal|
|             9|    109|   950|SUCCESS|           Large|
|            11|    111|   700|SUCCESS|          Normal|
|            13|    113|  1000|SUCCESS|           Large|
|            15|    115|   850|SUCCESS|           Large|
+--------------+-------+------+-------+----------------+



In [32]:
exp_df = spark.read.format("csv").option("header", True).option("inferSchema" , True).load("/opt/spark-data/employees_experience.csv")
exp_df.show()
# exp_df.printSchema()

+------+-------+------------+
|emp_id|   name|joining_year|
+------+-------+------------+
|     1|   John|        2018|
|     2|  Alice|        2023|
|     3|    Bob|        2015|
|     4|  David|        2025|
|     5|   Emma|        2020|
|     6|Michael|        2012|
|     7| Sophia|        2019|
|     8|  James|        2024|
|     9| Olivia|        2017|
|    10|William|        2010|
|    11|   Noah|        2022|
|    12|    Ava|        2016|
+------+-------+------------+



In [38]:
exp_df.withColumn('experience' , lit('2026') - col('joining_year')).withColumn('experience_band' , when((col('experience')>=0) & (col('experience')<=2 ), "Fresher")\
                                                                               .when((col('experience')>=3) & (col('experience')<=7) , "Mid")\
                                                                               .when(col('experience')>=8 , "Senior")
                                                                               .otherwise("Invalid")).show()

+------+-------+------------+----------+---------------+
|emp_id|   name|joining_year|experience|experience_band|
+------+-------+------------+----------+---------------+
|     1|   John|        2018|       8.0|         Senior|
|     2|  Alice|        2023|       3.0|            Mid|
|     3|    Bob|        2015|      11.0|         Senior|
|     4|  David|        2025|       1.0|        Fresher|
|     5|   Emma|        2020|       6.0|            Mid|
|     6|Michael|        2012|      14.0|         Senior|
|     7| Sophia|        2019|       7.0|            Mid|
|     8|  James|        2024|       2.0|        Fresher|
|     9| Olivia|        2017|       9.0|         Senior|
|    10|William|        2010|      16.0|         Senior|
|    11|   Noah|        2022|       4.0|            Mid|
|    12|    Ava|        2016|      10.0|         Senior|
+------+-------+------------+----------+---------------+



In [66]:
app_log_df = spark.read.format("csv").option("header", True).option("inferSchema" , True).load("/opt/spark-data/application_logs.csv")
app_log_df.show()

+------+--------------------+
|log_id|             message|
+------+--------------------+
|     1|ERROR: Database c...|
|     2|INFO: User login ...|
|     3|WARNING: Memory u...|
|     4|ERROR: File not f...|
|     5|INFO: Application...|
|     6|WARNING: CPU usag...|
|     7|ERROR: Network ti...|
|     8|INFO: User logout...|
|     9|WARNING: Disk spa...|
|    10|ERROR: Authentica...|
+------+--------------------+



In [70]:
app_log_df.withColumn('summary' , regexp_extract(
    col("message"),
    "^(ERROR|INFO|WARNING)",
    1
)).show()

[Stage 60:>                                                         (0 + 1) / 1]

+------+--------------------+-------+
|log_id|             message|summary|
+------+--------------------+-------+
|     1|ERROR: Database c...|  ERROR|
|     2|INFO: User login ...|   INFO|
|     3|WARNING: Memory u...|WARNING|
|     4|ERROR: File not f...|  ERROR|
|     5|INFO: Application...|   INFO|
|     6|WARNING: CPU usag...|WARNING|
|     7|ERROR: Network ti...|  ERROR|
|     8|INFO: User logout...|   INFO|
|     9|WARNING: Disk spa...|WARNING|
|    10|ERROR: Authentica...|  ERROR|
+------+--------------------+-------+



In [72]:
sales_df.withColumn('customer' , initcap(trim(col('customer')))).withColumn('customer_category' , when(col('amount')>=800 , 'Premium').otherwise("Normal")).show()

[Stage 61:>                                                         (0 + 1) / 1]

+--------+--------+------+-----------------+
|order_id|customer|amount|customer_category|
+--------+--------+------+-----------------+
|       1|    John|   500|           Normal|
|       2|   Alice|  1000|          Premium|
|       3|     Bob|   700|           Normal|
|       4|   David|   850|          Premium|
|       5|    Emma|   300|           Normal|
|       6|  Sophia|  1200|          Premium|
|       7|   James|   650|           Normal|
|       8|  Olivia|   950|          Premium|
|       9| William|   400|           Normal|
|      10|     Ava|  1100|          Premium|
+--------+--------+------+-----------------+



In [48]:
emp_pro = app_log_df = spark.read.format("csv").option("header", True).option("inferSchema" , True).load("/opt/spark-data/employees_promition.csv")
emp_pro.show()

+--------+----------+------+----------+
|employee|department|salary|experience|
+--------+----------+------+----------+
|    John|        IT| 90000|         5|
|   Alice|        HR| 40000|        10|
|     Bob|        IT| 60000|         2|
|   David|   Finance| 85000|         4|
|    Emma| Marketing| 45000|         7|
| Michael|        IT| 75000|         8|
|  Sophia|        HR| 30000|         3|
|   James|   Finance| 95000|         6|
|  Olivia| Marketing| 55000|         1|
| William|        IT| 82000|         5|
+--------+----------+------+----------+



In [49]:
emp_pro.withColumn('promotion_status' , when((col("salary")>80000) & (col("experience")>3) , "elegible").when(col("salary")<50000 , "not elegible").otherwise("review")).show()

[Stage 42:>                                                         (0 + 1) / 1]

+--------+----------+------+----------+----------------+
|employee|department|salary|experience|promotion_status|
+--------+----------+------+----------+----------------+
|    John|        IT| 90000|         5|        elegible|
|   Alice|        HR| 40000|        10|    not elegible|
|     Bob|        IT| 60000|         2|          review|
|   David|   Finance| 85000|         4|        elegible|
|    Emma| Marketing| 45000|         7|    not elegible|
| Michael|        IT| 75000|         8|          review|
|  Sophia|        HR| 30000|         3|    not elegible|
|   James|   Finance| 95000|         6|        elegible|
|  Olivia| Marketing| 55000|         1|          review|
| William|        IT| 82000|         5|        elegible|
+--------+----------+------+----------+----------------+



In [50]:
cust_df = app_log_df = spark.read.format("csv").option("header", True).option("inferSchema" , True).load("/opt/spark-data/customer_quality.csv")
cust_df.show()

[Stage 45:>                                                         (0 + 1) / 1]

+---+-----+---------------+-----+
| id| name|          email|  age|
+---+-----+---------------+-----+
|  1| John| john@gmail.com| 25.0|
|  2| NULL|  abc@gmail.com| 30.0|
|  3|Alice|           NULL| -5.0|
|  4|  Bob|  bob@gmail.com| 45.0|
|  5| NULL| mike@gmail.com|120.0|
|  6|Sarah|           NULL| 28.0|
|  7|David|david@gmail.com| 60.0|
|  8| Emma| emma@gmail.com| -1.0|
|  9| NULL| test@gmail.com| 35.0|
| 10| Alex| alex@gmail.com| 22.0|
+---+-----+---------------+-----+



In [55]:
cust_df.withColumn('name_status' , when(col('name').isNull() , "invalid").otherwise("valid"))\
.withColumn("email_status" ,when( col("email").isNull() , "Invalid").otherwise("valid"))\
.withColumn("age_status" , when((col("age")>=0) & (col("age")<100) , "valid").otherwise("invalid")).show()

[Stage 49:>                                                         (0 + 1) / 1]

+---+-----+---------------+-----+-----------+------------+----------+
| id| name|          email|  age|name_status|email_status|age_status|
+---+-----+---------------+-----+-----------+------------+----------+
|  1| John| john@gmail.com| 25.0|      valid|       valid|     valid|
|  2| NULL|  abc@gmail.com| 30.0|    invalid|       valid|     valid|
|  3|Alice|           NULL| -5.0|      valid|     Invalid|   invalid|
|  4|  Bob|  bob@gmail.com| 45.0|      valid|       valid|     valid|
|  5| NULL| mike@gmail.com|120.0|    invalid|       valid|   invalid|
|  6|Sarah|           NULL| 28.0|      valid|     Invalid|     valid|
|  7|David|david@gmail.com| 60.0|      valid|       valid|     valid|
|  8| Emma| emma@gmail.com| -1.0|      valid|       valid|   invalid|
|  9| NULL| test@gmail.com| 35.0|    invalid|       valid|     valid|
| 10| Alex| alex@gmail.com| 22.0|      valid|       valid|     valid|
+---+-----+---------------+-----+-----------+------------+----------+



In [61]:
raw_orders =spark.read.format("csv").option("header", True).option("inferSchema" , True).load("/opt/spark-data/raw_orders.csv")


In [64]:
raw_orders = raw_orders.withColumn('customer' , upper(trim(col('customer')))).withColumn('order_date' , to_date('order_date' , 'yyyy-MM-dd'))

In [65]:
raw_orders.printSchema()

root
 |-- order_id: integer (nullable = true)
 |-- customer: string (nullable = true)
 |-- amount: integer (nullable = true)
 |-- order_date: date (nullable = true)



In [77]:
cust_df =spark.read.format("csv").option("header", True).option("inferSchema" , True).load("/opt/spark-data/customer_quality.csv")
cust_df.withColumn('name_length' , length(col('name'))).withColumn("email_domain", regexp_extract(col("email") , "@(.*?)\\." , 1)).show()

+---+-----+---------------+-----+-----------+------------+
| id| name|          email|  age|name_length|email_domain|
+---+-----+---------------+-----+-----------+------------+
|  1| John| john@gmail.com| 25.0|          4|       gmail|
|  2| NULL|  abc@gmail.com| 30.0|       NULL|       gmail|
|  3|Alice|           NULL| -5.0|          5|        NULL|
|  4|  Bob|  bob@gmail.com| 45.0|          3|       gmail|
|  5| NULL| mike@gmail.com|120.0|       NULL|       gmail|
|  6|Sarah|           NULL| 28.0|          5|        NULL|
|  7|David|david@gmail.com| 60.0|          5|       gmail|
|  8| Emma| emma@gmail.com| -1.0|          4|       gmail|
|  9| NULL| test@gmail.com| 35.0|       NULL|       gmail|
| 10| Alex| alex@gmail.com| 22.0|          4|       gmail|
+---+-----+---------------+-----+-----------+------------+



In [78]:
#Q3 => idk

In [91]:
#Q5
df = spark.range(1)
df.select(date_format(to_date(lit('01-02-2026'), 'dd-MM-yyyy') , 'MM')).show()

+------------------------------------------------+
|date_format(to_date(01-02-2026, dd-MM-yyyy), MM)|
+------------------------------------------------+
|                                              02|
+------------------------------------------------+



In [94]:
prod_df =spark.read.format("csv").option("header", True).option("inferSchema" , True).load("/opt/spark-data/products.csv")
prod_df.withColumn('code-array' , split(col('code') , '-')).withColumn('Category' , col("code-array")[0]).withColumn('Object' , col('code-array')[1]).withColumn('num' , col('code-array')[2]).drop(col('code-array')).show()

+----------+-------------------+--------+---------+---+
|product_id|               code|Category|   Object|num|
+----------+-------------------+--------+---------+---+
|         1|    ELEC-LAPTOP-001|    ELEC|   LAPTOP|001|
|         2|      MOB-PHONE-002|     MOB|    PHONE|002|
|         3|     HOME-CHAIR-003|    HOME|    CHAIR|003|
|         4|   COMP-DESKTOP-004|    COMP|  DESKTOP|004|
|         5|   ACC-KEYBOARD-005|     ACC| KEYBOARD|005|
|         6|AUDIO-HEADPHONE-006|   AUDIO|HEADPHONE|006|
|         7|     FURN-TABLE-007|    FURN|    TABLE|007|
|         8|    SPORT-SHOES-008|   SPORT|    SHOES|008|
|         9|     BOOK-NOVEL-009|    BOOK|    NOVEL|009|
|        10|    WEAR-JACKET-010|    WEAR|   JACKET|010|
|        11|   ELEC-MONITOR-011|    ELEC|  MONITOR|011|
|        12|     MOB-TABLET-012|     MOB|   TABLET|012|
|        13|      HOME-LAMP-013|    HOME|     LAMP|013|
|        14|      ACC-MOUSE-014|     ACC|    MOUSE|014|
|        15|  AUDIO-SPEAKER-015|   AUDIO|  SPEAK

In [96]:
event_df =spark.read.format("csv").option("header", True).option("inferSchema" , True).load("/opt/spark-data/events.csv")
event_df.show()

+--------+-------------------+--------+
|event_id|         event_time|    user|
+--------+-------------------+--------+
|       1|2026/01/05 10:30:00|   john |
|       2|2026/02/10 18:45:00|   ALICE|
|       3|2026/03/15 09:20:00|    bob |
|       4|2026/04/01 12:15:00|  david |
|       5|2026/05/20 20:45:00|    EMMA|
|       6|2026/06/11 07:30:00| sophia |
|       7|2026/07/08 16:10:00|   JAMES|
|       8|2026/08/25 22:05:00| olivia |
|       9|2026/09/14 11:55:00| WILLIAM|
|      10|2026/10/30 19:25:00|    ava |
|      11|2026/11/05 08:40:00|    noah|
|      12|2026/12/15 14:30:00|    MIA |
+--------+-------------------+--------+



In [103]:
event_df.withColumn('event_time' , to_timestamp(col('event_time') , 'yyyy/MM/dd HH:mm:ss')).withColumn('user' , initcap(trim(col('user'))))\
.withColumn('event_year' , date_format(col('event_time') , 'yyyy'))\
.withColumn('eventMonth' , date_format(col('event_time') , 'MM'))\
.withColumn('event_day' , date_format(col('event_time') , 'dd'))\
.withColumn('hour' , date_format(col('event_time') , 'HH'))\
.withColumn('time_of_day' , when(col('hour') < 12 , "morning").when((col("hour") >=12 ) & (col("hour")<=17) , "afternoon").when(col("hour")>=18 , "evening")\
            .otherwise("invalid")).drop(col("hour")).show()
                                                                                                      

[Stage 96:>                                                         (0 + 1) / 1]

+--------+-------------------+-------+----------+----------+---------+-----------+
|event_id|         event_time|   user|event_year|eventMonth|event_day|time_of_day|
+--------+-------------------+-------+----------+----------+---------+-----------+
|       1|2026-01-05 10:30:00|   John|      2026|        01|       05|    morning|
|       2|2026-02-10 18:45:00|  Alice|      2026|        02|       10|    evening|
|       3|2026-03-15 09:20:00|    Bob|      2026|        03|       15|    morning|
|       4|2026-04-01 12:15:00|  David|      2026|        04|       01|  afternoon|
|       5|2026-05-20 20:45:00|   Emma|      2026|        05|       20|    evening|
|       6|2026-06-11 07:30:00| Sophia|      2026|        06|       11|    morning|
|       7|2026-07-08 16:10:00|  James|      2026|        07|       08|  afternoon|
|       8|2026-08-25 22:05:00| Olivia|      2026|        08|       25|    evening|
|       9|2026-09-14 11:55:00|William|      2026|        09|       14|    morning|
|   

In [98]:
from pyspark.sql.functions import *

df = event_df.withColumn(
    "event_time",
    to_timestamp(
        col("event_time"),
        "yyyy/MM/dd HH:mm:ss"
    )
)

df.show()

[Stage 92:>                                                         (0 + 1) / 1]

+--------+-------------------+--------+
|event_id|         event_time|    user|
+--------+-------------------+--------+
|       1|2026-01-05 10:30:00|   john |
|       2|2026-02-10 18:45:00|   ALICE|
|       3|2026-03-15 09:20:00|    bob |
|       4|2026-04-01 12:15:00|  david |
|       5|2026-05-20 20:45:00|    EMMA|
|       6|2026-06-11 07:30:00| sophia |
|       7|2026-07-08 16:10:00|   JAMES|
|       8|2026-08-25 22:05:00| olivia |
|       9|2026-09-14 11:55:00| WILLIAM|
|      10|2026-10-30 19:25:00|    ava |
|      11|2026-11-05 08:40:00|    noah|
|      12|2026-12-15 14:30:00|    MIA |
+--------+-------------------+--------+

